In [1]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import scipy.sparse as sp
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score
import pickle
        

In [4]:
# rebuild knowledge graph
triples = r"C:\Users\burit\Documents\GitHub\CADA\model\gene_hpo_withoutpatient\all.triples"
edges = []
with open(triples, 'r') as f:
    content = f.read().splitlines()
    content = [x.split('\t') for x in content]
    for triple in content:
        triple.pop(1)
        triple = tuple(triple)
        edges.append(triple)

G = nx.Graph()
G.add_edges_from(edges)
Gcc = sorted(nx.connected_components(G), key=len, reverse=True)
g = G.subgraph(Gcc[0])


In [5]:
from CADA.gae.preprocessing import mask_test_edges
np.random.seed(0) # make sure train-test split is consistent between notebooks
adj_sparse = nx.to_scipy_sparse_matrix(g)

# Perform train-validation-test split
adj_train, train_edges, train_edges_false, val_edges, val_edges_false, \
    test_edges, test_edges_false = mask_test_edges(adj_sparse, test_frac=.3, val_frac=.1, prevent_disconnect=False, verbose=True)
g_train = nx.from_scipy_sparse_matrix(adj_train) # new graph object with only non-hidden edges

preprocessing...
generating test/val sets...
creating false test edges...
creating false val edges...
creating false train edges...
final checks for disjointness...
creating adj_train...
Done with train-test split!



In [12]:
# Inspect train/test split
print("Total nodes:", adj_sparse.shape[0])
print("Total edges:", int(adj_sparse.nnz/2)) # adj is symmetric, so nnz (num non-zero) = 2*num_edges
print("Training edges (positive):", len(train_edges))
print("Training edges (negative):", len(train_edges_false))
print("Validation edges (positive):", len(val_edges))
print("Validation edges (negative):", len(val_edges_false))
print("Test edges (positive):", len(test_edges))
print("Test edges (negative):", len(test_edges_false))

Total nodes: 18879
Total edges: 183923
Training edges (positive): 110355
Training edges (negative): 110355
Validation edges (positive): 18392
Validation edges (negative): 18392
Test edges (positive): 55176
Test edges (negative): 55176


In [17]:
from node2vec import Node2Vec
from gensim.models import Word2Vec

# node2vec settings
# NOTE: When p = q = 1, this is equivalent to DeepWalk
dimensions = 128 
walk_length = 80 # Length of walk per source
num_walks = 10 # Number of walks per source
workers = 8 # Num. parallel workers
p = 1 # Return hyperparameter
q = 1 # In-out hyperparameter


In [ ]:
# Preprocessing, generate walks
g_n2v = Node2Vec(g_train, dimensions=dimensions, walk_length=walk_length, num_walks=num_walks, workers=workers, p=p, q=q) # create node2vec graph instance
g_n2v.preprocess_transition_probs()
walks = g_n2v.simulate_walks(NUM_WALKS, WALK_LENGTH)
walks = [map(str, walk) for walk in walks]

# Train skip-gram model
model = Word2Vec(walks, size=DIMENSIONS, window=WINDOW_SIZE, min_count=0, sg=1, workers=WORKERS, iter=ITER)

# Store embeddings mapping
emb_mappings = model.wv

Computing transition probabilities: 100%|██████████| 18879/18879 [02:08<00:00, 147.36it/s] 
